# 33 GPU sequence and video models

L4 推奨。structured sequence TCN、VideoMAE/DINO 系 frozen encoder、raw-video DNN fine-tune をまとめます。

各 stage は `BASE_DIR/reports/preflight/compact_runs/` に JSONL log を追記します。既に期待 artifact が揃っている stage は skip します。再実行したい場合は `FORCE_RERUN_ALL=True` にしてください。

In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import sys


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    try:
        drive.mount(str(mountpoint))
    except ValueError as exc:
        message = str(exc)
        if 'Mountpoint must not already contain files' in message or 'already mounted' in message:
            print(f'Drive mount warning: {message}')
            if not (mountpoint / 'MyDrive').exists():
                drive.mount(str(mountpoint), force_remount=True)
        else:
            raise
    except Exception as exc:
        print(f'Colab Drive mount skipped or failed: {exc}')


# v2 をデフォルトにします。v1 を再実行したい時だけここを書き換えてください。
os.environ.setdefault('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
os.environ.setdefault('BATTING_CODE_ROOT', '/content/drive/MyDrive/codex/batting_codex_handoff')


def _is_repo_dir(path: Path) -> bool:
    return (
        (path / 'src' / 'sport_pipeline' / '__init__.py').exists()
        and (path / 'configs').exists()
        and (path / 'notebooks').exists()
    )


def _resolve_repo_dir() -> Path:
    fixed = Path(os.environ['BATTING_CODE_ROOT'])
    mydrive = Path('/content/drive/MyDrive')
    candidates = [
        fixed,
        Path('/content/drive/MyDrive/codex/batting_codex_handoff'),
        Path('/content/drive/MyDrive/batting_codex_handoff'),
        Path.cwd(),
        *Path.cwd().parents,
    ]
    checked = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            return candidate
    raise FileNotFoundError(
        'sport_pipeline repo が見つかりません。Drive mount と repo 配置を確認してください。\n'
        f'BATTING_CODE_ROOT={fixed}\n'
        '推奨配置: /content/drive/MyDrive/codex/batting_codex_handoff\n'
        '別配置の場合は compact notebook の最初の cell より前に次を実行してください。\n'
        '  %env BATTING_CODE_ROOT=/content/drive/MyDrive/あなたの配置/batting_codex_handoff\n'
        f'MyDrive exists={mydrive.exists()}\n'
        '確認した候補:\n- ' + '\n- '.join(checked)
    )


_mount_drive_if_needed()
REPO_DIR = _resolve_repo_dir()
os.environ['BATTING_CODE_ROOT'] = str(REPO_DIR)
NOTEBOOK_DIR = REPO_DIR / 'notebooks'
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ['BASEBALL_VISION_RUN_PROFILE']
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME
src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(f'sport_pipeline を import できません: {src_dir}')

from sport_pipeline.compact_notebooks import CompactRunLogger
from sport_pipeline.io import read_table
from sport_pipeline.pipeline.run_profile import artifact_id, load_run_profile, run_id

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
FULL_RUN_ID = run_id(RUN_PROFILE, 'full_run_id')
CONTEXT_RUN_ID = run_id(RUN_PROFILE, 'context_run_id')
SEQUENCE_RUN_ID = run_id(RUN_PROFILE, 'sequence_run_id')
SEQUENCE_TCN_RUN_ID = run_id(RUN_PROFILE, 'sequence_tcn_run_id')
VIDEO_LIGHTWEIGHT_RUN_ID = run_id(RUN_PROFILE, 'video_lightweight_run_id')
VIDEO_FROZEN_RUN_ID = run_id(RUN_PROFILE, 'video_frozen_run_id')
VIDEO_FINETUNE_RUN_ID = run_id(RUN_PROFILE, 'video_finetune_run_id')
PLAYER_SEASON_RUN_ID = run_id(RUN_PROFILE, 'player_season_run_id')
VLM_RUN_ID = run_id(RUN_PROFILE, 'vlm_run_id')
FUSION_RUN_ID = run_id(RUN_PROFILE, 'fusion_run_id')
METHOD_EVALUATION_REPORT_ID = run_id(RUN_PROFILE, 'method_evaluation_report_id')
VIDEO_ABLATION_REPORT_ID = run_id(RUN_PROFILE, 'video_ablation_report_id')
STRUCTURED_SEQUENCE_FEATURE_ID = artifact_id(RUN_PROFILE, 'structured_sequence_feature_id')
CLIP_EMBEDDING_FEATURE_ID = artifact_id(RUN_PROFILE, 'clip_embedding_feature_id')
PLAYER_SEASON_EMBEDDING_FEATURE_ID = artifact_id(RUN_PROFILE, 'player_season_embedding_feature_id')
VIDEO_LIGHTWEIGHT_FEATURE_ID = artifact_id(RUN_PROFILE, 'video_lightweight_feature_id', 'video_lightweight_features_mlb_2024_2026_v2')
VIDEO_EMBEDDING_FEATURE_ID = artifact_id(RUN_PROFILE, 'video_embedding_feature_id')
VLM_FEATURE_ID = artifact_id(RUN_PROFILE, 'vlm_feature_id')

print('RUN_PROFILE =', RUN_PROFILE_NAME)
print('RUN_PROFILE_PATH =', RUN_PROFILE_PATH)
print('BASE_DIR =', BASE_DIR)
print('NOTEBOOK_DIR =', NOTEBOOK_DIR)
print('src_dir =', src_dir)


In [ ]:
COMPACT_RUN_NAME = '33_gpu_sequence_and_video_models'
LOGGER = CompactRunLogger(BASE_DIR, COMPACT_RUN_NAME, run_profile_name=RUN_PROFILE_NAME)
FORCE_RERUN_ALL = False

STRUCTURED_FRAMES_PATH = BASE_DIR / 'features' / STRUCTURED_SEQUENCE_FEATURE_ID / 'frames.parquet'
TCN_SUMMARY_PATH = BASE_DIR / f'reports/preflight/train_tcn_{SEQUENCE_TCN_RUN_ID}.json'

def _safe_read_rows(path):
    try:
        return read_table(path) if path.exists() else []
    except Exception as exc:
        print('WARNING: 33 completion check failed for', path, exc)
        return []

def _safe_read_json(path):
    try:
        return json.loads(path.read_text(encoding='utf-8')) if path.exists() else {}
    except Exception as exc:
        print('WARNING: 33 summary check failed for', path, exc)
        return {}

def _frame_pose_coverage(row):
    if row.get('pose_coverage') is not None:
        return float(row.get('pose_coverage') or 0.0)
    names = row.get('feature_names') or []
    values = row.get('feature_values') or []
    if isinstance(names, list) and isinstance(values, list) and 'pose_coverage' in names:
        index = names.index('pose_coverage')
        if index < len(values):
            return float(values[index] or 0.0)
    return 0.0

structured_frame_rows = _safe_read_rows(STRUCTURED_FRAMES_PATH)
sequence_pose_coverage_rows = sum(1 for row in structured_frame_rows if _frame_pose_coverage(row) > 0.0)
tcn_summary = _safe_read_json(TCN_SUMMARY_PATH)
tcn_summary_pose_rows = int(tcn_summary.get('input_pose_coverage_rows') or 0)
FORCE_SEQUENCE_TCN_FOR_POSE = sequence_pose_coverage_rows > 0 and tcn_summary_pose_rows == 0

print('sequence_pose_coverage_rows =', sequence_pose_coverage_rows)
print('tcn_summary_pose_rows =', tcn_summary_pose_rows)
print('force_sequence_tcn_for_pose =', FORCE_SEQUENCE_TCN_FOR_POSE)

RUN_SEQUENCE_TCN = True
RUN_FROZEN_VISUAL_ENCODER = True
RUN_RAW_VIDEO_FINETUNE = True

stages = [
    {
        'notebook': '18_sequence_tcn_training.ipynb',
        'label': 'TCN/MS-TCN structured sequence training',
        'enabled': RUN_SEQUENCE_TCN,
        'force': FORCE_RERUN_ALL or FORCE_SEQUENCE_TCN_FOR_POSE,
        'expected_outputs': [f'predictions/{SEQUENCE_TCN_RUN_ID}/predictions_v1.parquet', f'predictions/{SEQUENCE_TCN_RUN_ID}/metrics_v1.json', f'reports/preflight/train_tcn_{SEQUENCE_TCN_RUN_ID}.json', f'reports/preflight/train_tcn_{SEQUENCE_TCN_RUN_ID}_progress.json', f'models/sequence/{SEQUENCE_TCN_RUN_ID}/checkpoint.pt'],
        'progress_files': [f'reports/preflight/train_tcn_{SEQUENCE_TCN_RUN_ID}_progress.json'],
    },
    {
        'notebook': '19_frozen_visual_encoder.ipynb',
        'label': 'Frozen VideoMAE/DINO visual encoder',
        'enabled': RUN_FROZEN_VISUAL_ENCODER,
        'force': FORCE_RERUN_ALL,
        'expected_outputs': [f'features/{VIDEO_EMBEDDING_FEATURE_ID}/manifest.parquet', f'predictions/{VIDEO_FROZEN_RUN_ID}/predictions_v1.parquet', f'predictions/{VIDEO_FROZEN_RUN_ID}/metrics_v1.json', f'reports/preflight/frozen_visual_encoder_{VIDEO_FROZEN_RUN_ID}.json', f'reports/preflight/frozen_visual_encoder_{VIDEO_FROZEN_RUN_ID}_progress.json', f'models/video/{VIDEO_FROZEN_RUN_ID}/linear_head_checkpoint.pt'],
        'progress_files': [f'reports/preflight/frozen_visual_encoder_{VIDEO_FROZEN_RUN_ID}_progress.json'],
    },
    {
        'notebook': '19b_raw_video_finetune.ipynb',
        'label': 'Raw video DNN fine-tune',
        'enabled': RUN_RAW_VIDEO_FINETUNE,
        'force': FORCE_RERUN_ALL,
        'expected_outputs': [f'predictions/{VIDEO_FINETUNE_RUN_ID}/predictions_v1.parquet', f'predictions/{VIDEO_FINETUNE_RUN_ID}/metrics_v1.json', f'reports/preflight/raw_video_finetune_{VIDEO_FINETUNE_RUN_ID}.json', f'reports/preflight/raw_video_finetune_{VIDEO_FINETUNE_RUN_ID}_progress.json', f'models/video/{VIDEO_FINETUNE_RUN_ID}/checkpoint.pt'],
        'progress_files': [f'reports/preflight/raw_video_finetune_{VIDEO_FINETUNE_RUN_ID}_progress.json'],
    },
]
LOGGER.run_stages(ipython=get_ipython(), notebook_dir=NOTEBOOK_DIR, stages=stages)
